# 01 · Explore the raw docs
Loads the 495 raw ASP.NET Core docs and measures how messy they are, before any cleaning.
Every cell starts with a comment explaining what it does, and every line has a comment in plain language.

In [1]:
# ===== CELL 1: Load all the docs into a table =====
# Reads every .md file from data/raw and puts them into one pandas table
# (one row per file), so we can count and inspect them easily.

from pathlib import Path   # Path: a handy way to work with folders and file paths
import pandas as pd        # pandas: the library for working with tables (called DataFrames)
import re                  # re: "regular expressions", a mini-language for finding text patterns

RAW = Path("../data/raw")  # Where the raw docs live. ".." means "go up one folder" (from notebooks/ to the project folder)

rows = []                                   # An empty list; we'll add one entry per file to it
for p in RAW.rglob("*.md"):                 # Loop over every .md file in data/raw, including all subfolders ("r" = recursive)
    text = p.read_text(encoding="utf-8-sig")    # Read the whole file as text; "utf-8-sig" also removes an invisible character (BOM) some Windows files start with
    rows.append({                           # Add a small dictionary describing this file to our list
        "area": p.relative_to(RAW).parts[0],        # The first folder after data/raw: "blazor", "fundamentals" or "mvc"
        "path": p.relative_to(RAW).as_posix(),      # The file's path inside data/raw, written with "/" (e.g. "blazor/index.md")
        "chars": len(text),                         # How many characters the file has (its size)
        "text": text,                               # The full text of the file
    })

docs = pd.DataFrame(rows)                   # Turn the list of dictionaries into a table: one row per file, one column per key
print("Total documents:", len(docs))        # Print how many files we loaded
docs.groupby("area")["chars"].agg(["count", "sum", "mean", "max"]).round(0)   # For each area: number of files, total size, average size, biggest file. The last line of a cell is displayed automatically

Total documents: 495


,count,sum,mean,max
area,,,,
blazor,191,4169801,21831.0,108155
fundamentals,224,3240896,14468.0,240414
mvc,80,1109489,13869.0,106364


In [2]:
# ===== CELL 2: Look at one raw document =====
# Picks one file that contains version blocks and prints its beginning,
# so we can see the "mess" with our own eyes before cleaning it.

sample = docs[docs["text"].str.contains(":::moniker")].iloc[0]   # Keep only rows whose text contains ":::moniker", then take the first one (.iloc[0] = row number 0)
print(sample["path"])          # Print which file it is
print("=" * 80)                # Print 80 "=" characters as a separator line
print(sample["text"][:3000])   # Print the first 3,000 characters ([:3000] means "from the start up to character 3,000")

blazor/advanced-scenarios.md
---
title: ASP.NET Core Blazor advanced scenarios (render tree construction)
description: Learn how to incorporate manual logic for building Blazor render trees (RenderTreeBuilder).
monikerRange: '>= aspnetcore-3.1'
ms.date: 11/11/2025
---
# ASP.NET Core Blazor advanced scenarios (render tree construction)

[!INCLUDE[](~/includes/not-latest-version.md)]

This article describes the advanced scenario for building Blazor render trees manually with <xref:Microsoft.AspNetCore.Components.Rendering.RenderTreeBuilder>.

> [!WARNING]
> Use of <xref:Microsoft.AspNetCore.Components.Rendering.RenderTreeBuilder> to create components is an *advanced scenario*. A malformed component (for example, an unclosed markup tag) can result in undefined behavior. Undefined behavior includes broken content rendering, loss of app features, and ***compromised security***.

## Manually build a render tree (`RenderTreeBuilder`)

<xref:Microsoft.AspNetCore.Components.Rendering.RenderTree

In [3]:
# ===== CELL 3: Count the messy patterns =====
# For each kind of "mess" we expect, counts how many docs contain it.
# This tells us which cleaning steps matter most.

patterns = {                                             # A dictionary: a readable name -> the text pattern (regex) that finds it
    "YAML header (--- at top)": r"\A---\s*\n",             # \A = the very start of the file, then "---", optional spaces (\s*), then a line break (\n)
    "Version blocks (:::moniker)": r":::moniker",          # The marker Microsoft uses for version-specific text
    "Includes ([!INCLUDE])": r"\[!INCLUDE",                 # A line that pulls in text from another file. "\[" means a real "[" (a plain "[" has a special meaning in regex)
    "External code refs (:::code)": r":::code",            # A pointer to a code file stored somewhere else
    "Cross-refs (<xref:)": r"<xref:",                      # A link to an API page, like <xref:Microsoft.AspNetCore...>
    "Note/warning boxes": r"> \[!(?:NOTE|WARNING|IMPORTANT|TIP|CAUTION)\]",   # Callout boxes; "(?:A|B|C)" means "A or B or C"
    "Images": r"!\[",                                      # Markdown images start with "!["
    "Code blocks (```)": r"```",                           # Code blocks are fenced with three backticks
}

summary = pd.DataFrame({                                 # Build a small table with one column...
    "Docs containing it": {name: docs["text"].str.contains(pat, regex=True).sum() for name, pat in patterns.items()}   # ...for each pattern, check every doc (True/False) and add up the Trues = number of docs containing it
})
summary["% of docs"] = (summary["Docs containing it"] / len(docs) * 100).round(0)   # Add a second column: the same count as a percentage of all docs
summary                                                  # Display the table

,Docs containing it,% of docs
YAML header (--- at top),338,68.0
Version blocks (:::moniker),256,52.0
Includes ([!INCLUDE]),284,57.0
External code refs (:::code),126,25.0
Cross-refs (<xref:),306,62.0
Note/warning boxes,251,51.0
Images,51,10.0
Code blocks (```),342,69.0


In [4]:
# ===== CELL 4: Which ASP.NET Core versions do the docs cover? =====
# Finds every version range written in the docs (like ">= aspnetcore-8.0")
# and counts how often each one appears.

versions = (
    docs["text"]
    .str.findall(r':::moniker range="([^"]+)"')   # From every doc, grab the text inside range="..." (the part in brackets is what we keep). Gives a list per doc
    .explode()                                     # Turn each doc's list into separate rows, one range per row
    .value_counts()                                # Count how many times each distinct range appears
)
versions.head(15)   # Show the 15 most common ranges

text
>= aspnetcore-8.0                     385
>= aspnetcore-9.0                     258
>= aspnetcore-6.0                     230
< aspnetcore-8.0                      224
< aspnetcore-6.0                      192
>= aspnetcore-7.0 < aspnetcore-8.0    179
>= aspnetcore-8.0 < aspnetcore-9.0    174
< aspnetcore-5.0                      167
>= aspnetcore-6.0 < aspnetcore-7.0    154
>= aspnetcore-5.0 < aspnetcore-6.0    142
>= aspnetcore-11.0                    110
>= aspnetcore-10.0                     90
>= aspnetcore-6.0 < aspnetcore-8.0     68
>= aspnetcore-5.0                      61
>= aspnetcore-7.0                      51
Name: count, dtype: int64

In [5]:
# ===== CELL 5: Find tiny files =====
# Very small files are often fragments or stubs rather than real articles.

tiny = docs[docs["chars"] < 800].sort_values("chars")   # Keep only files shorter than 800 characters, sorted from smallest to largest
print("Files under 800 characters:", len(tiny))          # Print how many there are
tiny[["path", "chars"]].head(15)                         # Show the path and size of the 15 smallest

Files under 800 characters: 42


,path,chars
458,mvc/models/file-uploads/samples/5.x/README.md,29
263,fundamentals/aot/includes/aot_preview.md,74
404,fundamentals/servers/httpsys/samples/3.x/Sampl...,246
403,fundamentals/servers/httpsys/samples/2.x/Sampl...,246
405,fundamentals/servers/httpsys/samples/8.x/Sampl...,246
279,fundamentals/app-state/samples/2.x/SessionSamp...,248
289,fundamentals/file-providers/samples/2.x/FilePr...,251
290,fundamentals/file-providers/samples/3.x/FilePr...,251
350,fundamentals/openapi/includes/using-openapi-do...,254
197,blazor/security/includes/wasm-aad-b2c-custom-p...,271


In [6]:
# ===== CELL 6: Where do the includes point? =====
# Lists every [!INCLUDE] reference and checks whether it points inside
# our downloaded folders or somewhere else.

inc = (
    docs["text"]
    .str.findall(r"\[!INCLUDE\s*\[[^\]]*\]\(([^)]+)\)\]")   # From every doc, grab the file path inside [!INCLUDE[...](PATH)]
    .explode()                                               # One row per include reference
    .dropna()                                                # Remove empty rows (docs that have no includes)
)

print("Total include references:", len(inc))                                              # How many include references exist in total
print("\nPoints outside our folders (starts with ~/):", inc.str.startswith("~/").sum())   # "~/" means "from the docs root"; count those. "\n" prints a blank line first
print("Relative paths (likely inside our folders):", (~inc.str.startswith("~/")).sum())   # A "~" in front of a True/False column flips it, so this counts paths NOT starting with "~/"
print("\nMost common includes:")                                                           # A heading for the next output
inc.value_counts().head(15)                                                                # The 15 most-used include files and how often each appears

Total include references: 699

Points outside our folders (starts with ~/): 693
Relative paths (likely inside our folders): 6

Most common includes:


text
~/includes/not-latest-version.md                                  198
~/includes/aspnetcore-repo-ref-source-links.md                     65
~/includes/package-reference.md                                    57
~/includes/not-latest-version-without-not-supported-content.md     30
~/includes/default-launch-profile-for-dotnet-cli.md                18
~/includes/regex.md                                                18
~/blazor/security/includes/secure-authentication-flows.md          16
~/includes/azure-active-directory-b2c-eol-support-notice.md        13
~/blazor/includes/prefer-exact-matches.md                           8
~/blazor/tutorials/movie-database-app/includes/troubleshoot.md      8
~/includes/trailers.md                                              8
~/includes/reset.md                                                 8
~/blazor/security/includes/remote-authentication-paths.md           7
~/blazor/security/includes/redirecttologin-component.md             7
~/blazor/securi

In [7]:
# ===== CELL 7: Are the files without a header include fragments? =====
# Real articles start with a "---" header; this checks what the others are.

has_yaml = docs["text"].str.contains(r"\A---\s*\n", regex=True)   # True/False for every doc: does it start with a "---" header?
no_yaml = docs[~has_yaml]                                         # Keep only the docs WITHOUT a header ("~" flips True/False)

print("Files without YAML header:", len(no_yaml))                                                          # How many have no header
print("...of which are inside an 'includes' folder:", no_yaml["path"].str.contains("/includes/").sum())   # How many of those live in an "includes" folder
no_yaml["path"].head(15)                                                                                   # Show the first 15 of them

Files without YAML header: 157
...of which are inside an 'includes' folder: 85


112               blazor/includes/closure-of-circuits.md
113    blazor/includes/compression-with-untrusted-dat...
114                    blazor/includes/default-scheme.md
115              blazor/includes/prefer-exact-matches.md
116                      blazor/includes/prerendering.md
174     blazor/includes/js-interop/blazor-page-script.md
175         blazor/includes/js-interop/js-collocation.md
176    blazor/includes/js-interop/synchronous-js-inte...
177    blazor/includes/js-interop/synchronous-js-inte...
178    blazor/security/includes/additional-scopes-sta...
179    blazor/security/includes/additional-scopes-sta...
180            blazor/security/includes/app-component.md
181    blazor/security/includes/authentication-compon...
182     blazor/security/includes/authorize-client-app.md
183      blazor/security/includes/fetchdata-component.md
Name: path, dtype: str